In [36]:
import numpy as np
from sklearn.utils import shuffle
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

X_speech = np.load(r"E:\Pythonfile\Voice-Activity-Detect\data/feature/train/lfcc_lbp_speech_20.npy")
X_non_1 = np.load(r"E:\Pythonfile\Voice-Activity-Detect\data/feature/train/lfcc_lbp_nonspeech_1_20.npy")
X_non_2 = np.load(r"E:\Pythonfile\Voice-Activity-Detect\data/feature/train/lfcc_lbp_nonspeech_2_20.npy")
X_non = np.concatenate([X_non_1, X_non_2], axis=0)

X_train = np.vstack([X_non, X_speech]).astype(np.float32)
y_train = np.hstack([
    np.zeros(len(X_non), dtype=np.int8),
    np.ones(len(X_speech), dtype=np.int8)
])

X_train, y_train = shuffle(X_train, y_train, random_state=42)
X_1 = np.load(r"E:/Pythonfile/Voice-Activity-Detect/data/feature/train/lfcc_lbp_speech_20_aurora.npy")
X_2 = np.load(r"E:\Pythonfile\Voice-Activity-Detect\data\feature\train\lfcc_lbp_speech-noise_20_aurora.npy")
X_test = np.vstack([X_1, X_2]).astype(np.float32)
y_test = np.hstack([
    np.ones(len(X_1), dtype=np.int8),
    np.ones(len(X_2), dtype=np.int8),
])
print("Final shape:", X_train.shape, y_test.shape)

Final shape: (126713, 256) (723,)


In [22]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import roc_curve


# ===== Hàm tính EER =====
def compute_eer(y_true, y_score):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    eer = fpr[np.nanargmin(np.absolute((fnr - fpr)))]
    return eer

spoof = np.sum(y == 0)
bonafide = np.sum(y == 1)

scale_pos_weight = spoof / bonafide

print("scale_pos_weight:", scale_pos_weight)


model = Pipeline([
    ("scaler", StandardScaler()),
    ("xgb", XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        n_jobs=-1,
        random_state=42
    ))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]

    # print(f"Accuracy: {accs[-1]}")
    # print(f"Precision: {precisions[-1]}")
    # print(f"Recall: {recalls[-1]}")
    # print(f"F1: {f1s[-1]}")
    # print(f"EER: {eers[-1]}")
    # print()

print("Recall:", recall_score(y_test, y_pred))
# 0.676791277258567

scale_pos_weight: 0.9793648562100692
Recall: 0.9319444444444445


In [37]:
from sklearn.ensemble import RandomForestClassifier

import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import roc_curve


# ===== Hàm tính EER =====
def compute_eer(y_true, y_score):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    eer = fpr[np.nanargmin(np.absolute((fnr - fpr)))]
    return eer

spoof = np.sum(y == 0)
bonafide = np.sum(y == 1)

scale_pos_weight = spoof / bonafide

print("scale_pos_weight:", scale_pos_weight)


model = Pipeline([
        ("scaler", StandardScaler()),
        ("rf", RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            class_weight="balanced",
            n_jobs=-1,
            random_state=42
        ))
    ])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]

    # print(f"Accuracy: {accs[-1]}")
    # print(f"Precision: {precisions[-1]}")
    # print(f"Recall: {recalls[-1]}")
    # print(f"F1: {f1s[-1]}")
    # print(f"EER: {eers[-1]}")
    # print()

print("Recall:", recall_score(y_test, y_pred))
# 0.676791277258567


scale_pos_weight: 0.9793648562100692
Recall: 0.7247579529737206
